<a href="https://colab.research.google.com/github/MXC66ai/MultimodalLLM-ObjectDetection/blob/main/MedVitEncoder_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np
from PIL import Image

###############################################
#               1.  Encoder
###############################################

class MedViTEncoder(nn.Module):
    """
    现有卷积式 2D + 3D Encoder
    """
    def __init__(self, embed_dim=64, num_heads=4, mlp_dim=128):
        super(MedViTEncoder, self).__init__()

        # ============ 2D 分支 ============
        self.layers_2d = nn.Sequential(
            nn.Conv2d(1, embed_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(embed_dim, embed_dim * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(embed_dim * 2, embed_dim * 4, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2)   # H,W → H/2, W/2
        )

        # ============ 3D 分支 ============
        self.layers_3d = nn.Sequential(
            nn.Conv3d(1, embed_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv3d(embed_dim, embed_dim * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv3d(embed_dim * 2, embed_dim * 4, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AvgPool3d(kernel_size=2)
        )

    def forward_2d(self, x):
        return self.layers_2d(x)

    def forward_3d(self, x):
        return self.layers_3d(x)


###############################################
#            2.  最小可运行融合模块
###############################################

class FusionModule(nn.Module):
    """
    最小可运行跨模态融合模块：
    """
    def __init__(self, in_channels_2d, in_channels_3d, out_channels):
        super(FusionModule, self).__init__()

        self.proj3d = nn.AdaptiveAvgPool3d((1, None, None))  # [B,C,D,H,W]→[B,C,1,H,W]
        self.fuse = nn.Sequential(
            nn.Conv2d(in_channels_2d + in_channels_3d, out_channels, kernel_size=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )

    def forward(self, f2d, f3d):
        # 3D → 2D (平均投影 D 维)
        f3d_2d = self.proj3d(f3d).squeeze(2)   # remove D dim → [B,C,H,W]

        # concat
        fused = torch.cat([f2d, f3d_2d], dim=1)
        return self.fuse(fused)


###############################################
#               3. Shared Decoder（已修复）
###############################################

class MedViTDecoder(nn.Module):
    """
    🌟 修复点：增加 Upsample 恢复回 128×128
    """
    def __init__(self):
        super(MedViTDecoder, self).__init__()

        self.decode_2d = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            # **关键修复：上采样恢复尺寸**
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),

            nn.Conv2d(64, 1, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.decode_2d(x)


###############################################
#            4. MedViT (主模型)
###############################################

class MedViT(nn.Module):
    """
    支持 2D + 3D 双输入的模型
    """
    def __init__(self):
        super(MedViT, self).__init__()

        self.encoder = MedViTEncoder(embed_dim=32)
        # 2D 输出通道 = 128
        # 3D 输出通道 = 128
        self.fusion = FusionModule(
            in_channels_2d=128,
            in_channels_3d=128,
            out_channels=128
        )
        self.decoder = MedViTDecoder()

    def forward(self, x2d, x3d):
        feat_2d = self.encoder.forward_2d(x2d)
        feat_3d = self.encoder.forward_3d(x3d)

        fused = self.fusion(feat_2d, feat_3d)
        out = self.decoder(fused)
        return out


###############################################
#       5. Dummy Dataset（保证可运行）
###############################################

class Dummy2D3DDataset(Dataset):
    def __init__(self, num_samples=20):
        self.num_samples = num_samples
        self.transform_2d = transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # ===== 生成伪 2D 图像 =====
        arr2d = np.random.rand(128, 128) * 255
        arr2d = arr2d.astype(np.uint8)
        img2d = Image.fromarray(arr2d)
        img2d = self.transform_2d(img2d)

        # ===== 生成伪 3D 图像 =====
        arr3d = np.random.rand(32, 128, 128).astype(np.float32)
        img3d = torch.tensor(arr3d).unsqueeze(0)  # [1,32,H,W]

        # target = 二值 2D mask
        target = (img2d > 0.5).float()

        return img2d, img3d, target


###############################################
#                 6. 训练示例
###############################################

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dataset = Dummy2D3DDataset(num_samples=10)
    dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

    model = MedViT().to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(2):
        for img2d, img3d, target in dataloader:
            img2d, img3d, target = img2d.to(device), img3d.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(img2d, img3d)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1}, Loss={loss.item():.5f}")

    print("Training complete.")


Epoch 1, Loss=0.61202
Epoch 2, Loss=0.58298
Training complete.


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np
from PIL import Image

###############################################
#               1.  Encoder
###############################################

class MedViTEncoder(nn.Module):
    """
    现有卷积式 2D + 3D Encoder
    """
    def __init__(self, embed_dim=64, num_heads=4, mlp_dim=128):
        super(MedViTEncoder, self).__init__()

        # ============ 2D 分支 ============
        self.layers_2d = nn.Sequential(
            nn.Conv2d(1, embed_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(embed_dim, embed_dim * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(embed_dim * 2, embed_dim * 4, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2)   # H,W → H/2, W/2
        )

        # ============ 3D 分支 ============
        self.layers_3d = nn.Sequential(
            nn.Conv3d(1, embed_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv3d(embed_dim, embed_dim * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv3d(embed_dim * 2, embed_dim * 4, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AvgPool3d(kernel_size=2)
        )

    def forward_2d(self, x):
        return self.layers_2d(x)

    def forward_3d(self, x):
        return self.layers_3d(x)


###############################################
#            2.  最小可运行融合模块
###############################################

class FusionModule(nn.Module):
    """
    最小可运行跨模态融合模块：
    feat_2d: [B, C2, H, W]
    feat_3d: [B, C3, D, H, W] -> average pool D -> [B,C3,H,W]
    融合方式：concat -> 1x1 conv -> BN -> ReLU
    """
    def __init__(self, in_channels_2d, in_channels_3d, out_channels):
        super(FusionModule, self).__init__()

        self.proj3d = nn.AdaptiveAvgPool3d((1, None, None))  # [B,C,D,H,W]→[B,C,1,H,W]
        self.fuse = nn.Sequential(
            nn.Conv2d(in_channels_2d + in_channels_3d, out_channels, kernel_size=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )

    def forward(self, f2d, f3d):
        # 3D -> 2D (平均投影 D 维)
        f3d_2d = self.proj3d(f3d).squeeze(2)   # remove D dim -> [B,C,H,W]
        fused = torch.cat([f2d, f3d_2d], dim=1)
        return self.fuse(fused)


###############################################
#               3. Shared Decoder（含上采样）
###############################################

class MedViTDecoder(nn.Module):
    """
    修复点：增加 Upsample 恢复回 128x128
    """
    def __init__(self):
        super(MedViTDecoder, self).__init__()

        self.decode_2d = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            # 上采样恢复尺寸（encoder 做了 AvgPool2d）
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),

            nn.Conv2d(64, 1, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.decode_2d(x)


###############################################
#            4. MedViT (主模型)
###############################################

class MedViT(nn.Module):
    """
    支持 2D + 3D 双输入的模型
    """
    def __init__(self):
        super(MedViT, self).__init__()

        self.encoder = MedViTEncoder(embed_dim=32)
        # 2D 输出通道 = 32*4 = 128
        # 3D 输出通道 = 32*4 = 128
        self.fusion = FusionModule(
            in_channels_2d=128,
            in_channels_3d=128,
            out_channels=128
        )
        self.decoder = MedViTDecoder()

    def forward(self, x2d, x3d):
        feat_2d = self.encoder.forward_2d(x2d)
        feat_3d = self.encoder.forward_3d(x3d)
        fused = self.fusion(feat_2d, feat_3d)
        out = self.decoder(fused)
        return out


###############################################
#       5. Dummy Dataset（保证可运行）
###############################################

class Dummy2D3DDataset(Dataset):
    def __init__(self, num_samples=20):
        self.num_samples = num_samples
        self.transform_2d = transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # ===== 生成伪 2D 图像 =====
        arr2d = np.random.rand(128, 128) * 255
        arr2d = arr2d.astype(np.uint8)
        img2d = Image.fromarray(arr2d)
        img2d = self.transform_2d(img2d)  # [1,128,128]

        # ===== 生成伪 3D 图像 =====
        arr3d = np.random.rand(32, 128, 128).astype(np.float32)
        img3d = torch.tensor(arr3d).unsqueeze(0)  # [1,32,H,W]

        # target = 二值 2D mask
        target = (img2d > 0.5).float()

        return img2d, img3d, target


###############################################
#        6. Dice / IoU 指标函数（按 batch 逐样本计算）
###############################################

def dice_coeff_batch(pred, target, eps=1e-6):
    """
    pred, target: tensors with shape [B,1,H,W], values in [0,1]
    returns mean dice over batch
    """
    pred_bin = (pred > 0.5).float()
    target_bin = (target > 0.5).float()
    # flatten per sample
    pred_flat = pred_bin.view(pred_bin.size(0), -1)
    target_flat = target_bin.view(target_bin.size(0), -1)

    intersection = (pred_flat * target_flat).sum(dim=1)
    sums = pred_flat.sum(dim=1) + target_flat.sum(dim=1)
    dice = (2.0 * intersection + eps) / (sums + eps)
    return dice.mean().item()


def iou_score_batch(pred, target, eps=1e-6):
    """
    returns mean IoU over batch
    """
    pred_bin = (pred > 0.5).float()
    target_bin = (target > 0.5).float()
    pred_flat = pred_bin.view(pred_bin.size(0), -1)
    target_flat = target_bin.view(target_bin.size(0), -1)

    intersection = (pred_flat * target_flat).sum(dim=1)
    union = pred_flat.sum(dim=1) + target_flat.sum(dim=1) - intersection
    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()


###############################################
#                 7. 训练 + 评估示例
###############################################

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dataset = Dummy2D3DDataset(num_samples=50)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

    model = MedViT().to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    num_epochs = 3
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        count = 0
        for img2d, img3d, target in dataloader:
            img2d, img3d, target = img2d.to(device), img3d.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(img2d, img3d)  # [B,1,128,128]
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * img2d.size(0)
            count += img2d.size(0)

        epoch_loss = running_loss / count

        # ---------- evaluation after epoch ----------
        model.eval()
        dices = []
        ious = []
        with torch.no_grad():
            for img2d, img3d, target in dataloader:
                img2d, img3d, target = img2d.to(device), img3d.to(device), target.to(device)
                pred = model(img2d, img3d)
                dices.append(dice_coeff_batch(pred, target))
                ious.append(iou_score_batch(pred, target))

        mean_dice = float(np.mean(dices))
        mean_iou = float(np.mean(ious))

        print(f"Epoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.6f} - Dice: {mean_dice:.4f} - IoU: {mean_iou:.4f}")

    print("Training + Evaluation complete.")


Epoch 1/3 - Loss: 0.611844 - Dice: 0.6672 - IoU: 0.5006
Epoch 2/3 - Loss: 0.547904 - Dice: 0.7457 - IoU: 0.5945
Epoch 3/3 - Loss: 0.518022 - Dice: 0.7130 - IoU: 0.5540
Training + Evaluation complete.
